# Phase 9 Submission E (Submission D plus d3 as the seventh member)

# Phase 8 Submission D

Prep v2 artifacts at **9 slice groups**, SIX configs x five folds, probability mean over all
thirty models.

One change against the scored 0.921 (Submission B, c2+c3+f1 at 9 groups): **s2, c2's six-slot
attention config read at 336px, as a fourth member**, confirmed on five folds (pooled OOF 0.8538;
added to c2+c3+f1 at 5 groups +0.0042 [+0.0033, +0.0051], CI excluding zero; gold -0.0010, spanning
zero). It reads 9 of the nine stored groups at the stored 336px, so it needs no resize and
no second prep. Submission B's own two changes, kept as they were measured:

- **f1, ConvNeXt-tiny, as a third member.** Added to c2+c3: +0.0038 [+0.0028, +0.0049].
- **Averaging over slice windows.** Every member reads all 9 stored groups instead of 5:
  c2+c3 +0.0045 [+0.0034, +0.0055]; f1 alone +0.0053 [+0.0041, +0.0065].

Together, offline: pooled OOF 0.8641 -> 0.8713, +0.0072 [+0.0057, +0.0087];
gold 0.8919 -> 0.9059, +0.0140 [+0.0055, +0.0243].

**One prep pass still serves every member.** `prep_slots(n_groups=9, window_groups=5)` takes
each series' intensity window from the five training groups, so groups 0,2,4,6,8 of the
artifact are the 5-group training artifact pixel for pixel (checked on 800 corpus studies,
max difference 0). A member reading 5 groups gets exactly what it trained on; a member reading
9 gets four more windows on the same scale. Without `window_groups` the extra slices would
move the percentile window and rescale everything -- which is why the src guard below asserts
the parameter exists.

The 0.5 fallback is unchanged and remains non-negotiable: a submission that raises on one
hidden study scores zero.


In [ ]:
import glob, os, shutil, sys, tempfile, time

GIT_SHA = 'phase9-submit-f'

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]
# name, checkpoint pattern, backbone, head, slots read (of the six), read size, groups read.
# All three trained on the SAME prep v2 artifact, so one prep pass serves all of them.
MEMBERS = [
    dict(name='c2', pattern='c2_6slot_attn_fold*.pt', backbone='efficientnet_b0',
         head='slot_attention', n_slots=6, read=224, groups=9),
    dict(name='c3', pattern='c3_dinov2_2slot_fold*.pt',
         backbone='vit_small_patch14_dinov2.lvd142m', head='mean', n_slots=2, read=224, groups=9),
    dict(name='f1', pattern='f1_convnext_tiny_6slot_attn_fold*.pt',
         backbone='convnext_tiny.in12k_ft_in1k', head='slot_attention', n_slots=6, read=224,
         groups=9),
    # read=336: the artifact's stored size, which is what s2 trained on
    dict(name='s2', pattern='s2_c2_336_b4x2_fold*.pt', backbone='efficientnet_b0',
         head='slot_attention', n_slots=6, read=336, groups=9),
    dict(name='t2', pattern='t2_c2_blend3_fold*.pt', backbone='efficientnet_b0',
         head='slot_attention', n_slots=6, read=224, groups=9),
    # the only member with the group-token head; its loader shape is c2's
    dict(name='a2', pattern='a2_c2_grpattn_rndwin_fold*.pt', backbone='efficientnet_b0',
         head='group_attention', n_slots=6, read=224, groups=9),
    # a2's config trained on the three-source blend target (Screen D / d3 confirm)
    dict(name='d3', pattern='d3_a2blend_grpattn_rndwin_fold*.pt', backbone='efficientnet_b0',
         head='group_attention', n_slots=6, read=224, groups=9),
    # the same config read at 336px, the artifact's stored size (as s2)
    dict(name='d3_336', pattern='d3_336_a2blend_grpattn_rndwin_b4x2_fold*.pt', backbone='efficientnet_b0',
         head='group_attention', n_slots=6, read=336, groups=9),
]
for _m in MEMBERS:
    _m['ckpts'] = sorted(glob.glob(f"/kaggle/input/**/{_m['pattern']}", recursive=True))
    assert len(_m['ckpts']) == 5, (
        f"{_m['name']}: expected 5 fold checkpoints matching {_m['pattern']}, "
        f"found {len(_m['ckpts'])}: {_m['ckpts']}")
CKPTS = [c for _m in MEMBERS for c in _m['ckpts']]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

# A stale src here does not crash -- it silently prepares different pixels from the ones
# the checkpoints were trained on, and the only symptom is a lower score.
import inspect
from knee.dicom import SLOTS
from knee.model import KneeModel
from knee.prep import prep_slots
assert len(SLOTS) == 6, 'src slot table is not the six-slot layout these checkpoints trained on'
assert 'backbone_kwargs' in inspect.signature(KneeModel.__init__).parameters, (
    'src predates img_size passthrough, which the DINOv2 member needs')
# without window_groups a 9-group prep rescales the five training groups (a mean 1.2 grey
# levels, max 67, measured on the corpus) -- no crash, just different pixels
assert 'window_groups' in inspect.signature(prep_slots).parameters, 'src predates window_groups'
# f1 trained with absent slots packed out of the backbone
assert hasattr(KneeModel, '_backbone_features'), 'src predates slot packing'
print('src carries window_groups, packing and all three heads |', len(CKPTS), 'checkpoints across',
      len(MEMBERS), 'members')

In [ ]:
import numpy as np
import pandas as pd
import torch

from knee.dataset import PreppedSlotDataset, StudyDecodeError
from knee.infer import LABEL_COLUMNS, build_submission
from knee.prep import save_study_npz

# The training artifact's prep parameters, stored at 9 groups windowed on its five.
PREP_GROUPS, WINDOW_GROUPS, PREP_CROP_MM, PREP_SIZE = 9, 5, 130.0, 336
SLOT_NAMES = [n for n, _, _ in SLOTS]
PREP_SCHEME = 'v2'

test_df = pd.read_csv(f'{COMP_DIR}/test.csv')
test_series_df = pd.read_csv(f'{COMP_DIR}/test_series.csv')
TEST_ROOT = f'{COMP_DIR}/test_series'
print(f'{len(test_df)} test studies')

# Probe rather than assert: a hard GPU requirement turns an odd accelerator
# assignment into a zero on the scored run.
device = 'cpu'
if torch.cuda.is_available():
    try:
        _ = torch.zeros(1, device='cuda') + 1
        device = 'cuda'
    except Exception as exc:
        print(f'GPU present but unusable: {exc}')
print('device:', device)

for member in MEMBERS:
    extra = {'img_size': member['read']} if member['backbone'].startswith('vit_') else {}
    member['slots'] = SLOT_NAMES[:member['n_slots']]
    member['models'] = []
    for path in member['ckpts']:
        m = KneeModel(backbone_name=member['backbone'], num_labels=len(LABEL_COLUMNS),
                      pretrained=False, head=member['head'], **extra)
        m.load_state_dict(torch.load(path, map_location=device, weights_only=True))
        member['models'].append(m.to(device).eval())
    print(f"{member['name']}: {len(member['models'])} folds, {member['backbone']}, "
          f"{member['head']}, {member['n_slots']} slots at {member['read']}px, "
          f"{member['groups']} groups")
n_models = sum(len(m['models']) for m in MEMBERS)
print(f'{n_models} models total')

In [ ]:
NPZ_DIR = tempfile.mkdtemp(prefix='knee_prep_')
prep_seconds, forward_seconds = [], []
lat_routes, slot_fill, member_skips = {}, {}, {}

def predict_one(study_uid):
    """ONE prep_slots at 9 windowed groups -> npz -> each member reads it with its own
    slots, size and group count -> probability mean over every model of every member.
    Anything raising in here becomes a row of 0.5 via build_submission."""
    t0 = time.time()
    slot_slices, meta = prep_slots(
        study_uid, TEST_ROOT, test_series_df,
        n_groups=PREP_GROUPS, window_groups=WINDOW_GROUPS, crop_mm=PREP_CROP_MM,
        out_size=PREP_SIZE, slot_scheme=PREP_SCHEME)
    npz_path = os.path.join(NPZ_DIR, f'{study_uid}.npz')
    save_study_npz(npz_path, slot_slices, meta)
    prep_seconds.append(time.time() - t0)
    lat_routes[meta.get('route')] = lat_routes.get(meta.get('route'), 0) + 1
    slot_fill[len(slot_slices)] = slot_fill.get(len(slot_slices), 0) + 1

    try:
        t1 = time.time()
        per_model = []
        for member in MEMBERS:
            # a fresh read per member: slot subset, read size and group count are all part
            # of what each member was measured with
            dataset = PreppedSlotDataset([study_uid], NPZ_DIR, n_groups=member['groups'],
                                         out_size=member['read'], slots=member['slots'])
            try:
                image, mask, _, _ = dataset[0]
            except StudyDecodeError:
                # a two-slot member finding none of its slots must not cost the study its
                # other members' predictions. Every training study has a sagittal or
                # coronal fluid series, so this is insurance for the hidden set only.
                member_skips[member['name']] = member_skips.get(member['name'], 0) + 1
                continue
            batch = image.unsqueeze(0).to(device)
            mask_batch = mask.unsqueeze(0).to(device)
            with torch.no_grad():
                for m in member['models']:
                    per_model.append(torch.sigmoid(m(batch, mask=mask_batch))[0].cpu().numpy())
        if not per_model:
            raise StudyDecodeError(f'{{study_uid}}: no member could read a slot')
        # probability mean, flat over thirty models, so the six members weigh equally at
        # five folds each -- the weighting the offline estimate used. Rank mean measured
        # lower for f1 on both the screen and the confirm.
        probs = np.mean(per_model, axis=0)
        forward_seconds.append(time.time() - t1)
        return probs
    finally:
        os.remove(npz_path)

t0 = time.time()
submission = build_submission(test_df['StudyInstanceUID'].astype(str).tolist(), predict_one)
elapsed = time.time() - t0
print(f'{elapsed / 60:.1f} min for {len(submission)} studies '
      f'({elapsed / max(len(submission), 1):.2f} s/study)')
print(f'prep    {np.mean(prep_seconds):.3f}s/study (p95 {np.percentile(prep_seconds, 95):.3f})')
print(f'forward {np.mean(forward_seconds):.3f}s/study for {n_models} models '
      f'across {len(MEMBERS)} members')
print(f'laterality routes: {lat_routes}')
print(f'slots filled per study: {dict(sorted(slot_fill.items()))}')
print(f'members skipped for lacking every slot they read: {member_skips or 0}')
fallbacks = int((submission[LABEL_COLUMNS] == 0.5).all(axis=1).sum())
print(f'0.5 fallbacks: {fallbacks} of {len(submission)}')

In [ ]:
submission.to_csv('submission.csv', index=False)
print(submission.head())

assert len(submission) == len(test_df), 'row count must match test studies exactly'
assert list(submission.columns) == ['StudyInstanceUID'] + LABEL_COLUMNS, 'column contract'
assert submission[LABEL_COLUMNS].isna().sum().sum() == 0, 'no NaNs allowed'
vals = submission[LABEL_COLUMNS].to_numpy()
assert (vals >= 0).all() and (vals <= 1).all(), 'probabilities must be in [0, 1]'

print(f'\n{"label":20s} {"mean pred":>10s}')
for label in LABEL_COLUMNS:
    print(f'{label:20s} {submission[label].mean():10.4f}')